# META-CXR — encoder inference sensitivity

This notebook does **not** train seven models. It loads the single validation-selected E123 checkpoint, runs BioViL-T + PubMedCLIP + SwinV2 once per held-out batch, and evaluates seven downstream encoder subsets by removing inactive token spans before MHCAC. The deltas are descriptive sensitivity measurements, not causal contributions.

In [ ]:
DATASET_SLUG = "phuong20052/mimic-cxr-jpg-dataset"
CHECKPOINT_GCS_BUCKET = "meta-cxr-checkpoints-phuongnm"  # bucket holding checkpoint_best.pth at its root
RESULT_DATASET_HANDLE = ""           # pre-created private owner/slug
REPO_REF = "main"  # resolve and record the exact evaluation commit after fetch
BATCH_SIZE = 2
NUM_WORKERS = 4
SEED = 42


In [ ]:
import os, pathlib, subprocess, sys
for name, value in {'DATASET_SLUG': DATASET_SLUG, 'CHECKPOINT_GCS_BUCKET': CHECKPOINT_GCS_BUCKET, 'RESULT_DATASET_HANDLE': RESULT_DATASET_HANDLE, 'REPO_REF': REPO_REF}.items():
    if not value:
        raise ValueError(f'{name} is required')
repo_dir = pathlib.Path('/kaggle/working/META-CXR-SMOKETEST')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/minhphuong150505/META-CXR-SMOKETEST.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--depth=1', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
if len(REPO_COMMIT) != 40:
    raise RuntimeError('Could not resolve an exact evaluation commit')
print('Evaluation source commit:', REPO_COMMIT)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
os.environ['PYTHONPATH'] = str(repo_dir) + os.pathsep + os.environ.get('PYTHONPATH', '')


In [ ]:
# Checkpoints contain non-weight provenance and RNG state.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'


In [ ]:
# Keep credential material outside /kaggle/working, which Kaggle publishes as output.
from smoke.runtime import load_kaggle_secrets
load_kaggle_secrets(('GCS_SERVICE_ACCOUNT', 'WANDB_API_KEY'), '/tmp/.meta-cxr-secrets')
print('Loaded required secrets into OS environment (values hidden).')


In [ ]:
import json
from smoke.runtime import environment_fingerprint, assert_two_t4, compatibility_matrix
before = environment_fingerprint()
print(json.dumps(before, indent=2, sort_keys=True))
assert_two_t4(before)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', 'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', 'hi-ml-multimodal==0.2.1'], check=True)
after = environment_fingerprint()
assert_two_t4(after)
print(json.dumps(compatibility_matrix(before, after), indent=2, sort_keys=True))


In [ ]:
from smoke.runtime import discover_dataset, load_dataset_manifest, write_runtime_env_config
from smoke.checkpoints import download_best_checkpoint
dataset_root = discover_dataset(DATASET_SLUG)
dataset_manifest, _, dataset_hash = load_dataset_manifest(dataset_root)
if dataset_manifest.get('status') != 'qa_passed':
    raise RuntimeError('Dataset manifest is not QA-passed')
write_runtime_env_config(dataset_root, '/kaggle/working/meta-cxr-sensitivity')
checkpoint = download_best_checkpoint(
    CHECKPOINT_GCS_BUCKET, '/kaggle/working/meta-cxr-sensitivity/checkpoint'
)
print('Downloaded validation-selected checkpoint from GCS:', checkpoint)


In [ ]:
import torch
from IPython.display import Markdown, display
checkpoint_meta = torch.load(checkpoint, map_location='cpu')
# identity is exactly {dataset_manifest_sha256, config_fingerprint} -- the fields
# training writes and gates resume on. source_commit is provenance only (stored
# top-level, NOT in identity), so the eval commit may legitimately differ from the
# training commit; record it, don't gate on it.
identity = checkpoint_meta.get('identity')
if not identity or identity.get('dataset_manifest_sha256') != dataset_hash:
    raise RuntimeError('Checkpoint dataset identity mismatch')
checkpoint_source_commit = checkpoint_meta.get('source_commit')
print('Checkpoint trained by commit', checkpoint_source_commit, '| eval commit', REPO_COMMIT)
result_dir = pathlib.Path('/kaggle/working/meta-cxr-sensitivity-results')
result_path = result_dir / 'encoder_sensitivity.json'
eval_env = os.environ.copy()
eval_env['PYTHONPATH'] = str(repo_dir) + os.pathsep + eval_env.get('PYTHONPATH', '')
# Replaces the earlier aggregate-only result so the upload cell publishes the
# Table-5-compatible output under the same verified filename.
subprocess.run([sys.executable, 'scripts/evaluate_encoder_sensitivity.py', '--cfg-path', 'pretraining/configs/stage1_smoke_2xt4.yaml', '--checkpoint', str(checkpoint), '--output', str(result_path), '--overwrite', '--source-commit', REPO_COMMIT, '--dataset-manifest-sha256', dataset_hash, '--config-fingerprint', identity['config_fingerprint'], '--batch-size', str(BATCH_SIZE), '--num-workers', str(NUM_WORKERS)], check=True, env=eval_env)
summary = json.loads(result_path.read_text())
print({'status': summary['status'], 'method': summary['method'], 'test_studies': summary['test_studies'], 'wall_seconds': summary['wall_seconds'], 'peak_vram_bytes': summary['peak_vram_bytes']})
# Match paper Table 5: the paper does not include the ViT+Swin-only E23 row.
paper_rows = (
    ('E1', '✓', '−', '−'),
    ('E2', '−', '✓', '−'),
    ('E3', '−', '−', '✓'),
    ('E12', '✓', '✓', '−'),
    ('E13', '✓', '−', '✓'),
    ('E123', '✓', '✓', '✓'),
)
lines = [
    '### Encoder ablation — Paper Table 5 protocol',
    '',
    '| RN50 | ViT | Swin | Mean weighted F1 (3-class) |',
    '|:---:|:---:|:---:|---:|',
]
for run_id, rn50, vit, swin in paper_rows:
    score = summary['reports'][run_id]['paper_table5']['mean_weighted_f1']
    lines.append(f'| {rn50} | {vit} | {swin} | {score:.3f} |')
display(Markdown('\n'.join(lines)))
print('Protocol:', summary['paper_table5_protocol'])


In [ ]:
from smoke.checkpoints import upload_private_results_dataset
upload_private_results_dataset(RESULT_DATASET_HANDLE, result_dir)
print('Private aggregate result upload and manifest verification succeeded; local files retained.')
